# PySpark Test Notebook
This notebook verifies that PySpark is correctly installed and configured in the environment. It runs a few basic PySpark operations including creating a Spark session, generating data, applying transformations, performing aggregations, and running SQL queries.

In [1]:
import os
import sys

# Set environment variables for PySpark to use the current Python executable on Windows
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

print(f"Python Version: {sys.version}")
print(f"Python Executable: {sys.executable}")

Python Version: 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
Python Executable: c:\Users\NGUYEN\AppData\Local\Programs\Python\Python312\python.exe


## 1. Initialize Spark Session
We initialize a local Spark session. In a production environment, this would connect to a Spark cluster.

In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("ETS_PySpark_Test") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

print("Spark Session initialized successfully!")
print(f"Spark Version: {spark.version}")

Spark Session initialized successfully!
Spark Version: 4.1.2


## 2. Create Sample Data
We create some mock student enrollment data to test Spark DataFrames.

In [3]:
# Sample Data: (StudentID, Name, Age, Department, GPA)
data = [
    (101, "Nguyen Van A", 20, "Computer Science", 3.6),
    (102, "Tran Thi B", 22, "Computer Science", 3.8),
    (103, "Le Van C", 21, "Information Technology", 3.2),
    (104, "Pham Thi D", 23, "Business Administration", 3.9),
    (105, "Hoang Van E", 20, "Information Technology", 2.9),
    (106, "Ngo Thi F", 22, "Business Administration", 3.5)
]

columns = ["StudentID", "Name", "Age", "Department", "GPA"]

# Create DataFrame
df = spark.createDataFrame(data, schema=columns)

# Show Schema
df.printSchema()

# Show Data
df.show()

root
 |-- StudentID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Department: string (nullable = true)
 |-- GPA: double (nullable = true)

+---------+------------+---+--------------------+---+
|StudentID|        Name|Age|          Department|GPA|
+---------+------------+---+--------------------+---+
|      101|Nguyen Van A| 20|    Computer Science|3.6|
|      102|  Tran Thi B| 22|    Computer Science|3.8|
|      103|    Le Van C| 21|Information Techn...|3.2|
|      104|  Pham Thi D| 23|Business Administ...|3.9|
|      105| Hoang Van E| 20|Information Techn...|2.9|
|      106|   Ngo Thi F| 22|Business Administ...|3.5|
+---------+------------+---+--------------------+---+



## 3. Data Transformations & Filtering
Let's filter for students who are at least 21 years old and have a GPA >= 3.5.

In [4]:
# Filter and select columns
filtered_df = df.filter((col("Age") >= 21) & (col("GPA") >= 3.5)) \
                .select("Name", "Age", "Department", "GPA")

print("Filtered Students (Age >= 21 and GPA >= 3.5):")
filtered_df.show()

Filtered Students (Age >= 21 and GPA >= 3.5):
+----------+---+--------------------+---+
|      Name|Age|          Department|GPA|
+----------+---+--------------------+---+
|Tran Thi B| 22|    Computer Science|3.8|
|Pham Thi D| 23|Business Administ...|3.9|
| Ngo Thi F| 22|Business Administ...|3.5|
+----------+---+--------------------+---+



## 4. Aggregations (GroupBy)
Now, we group the students by department to compute the average GPA and the number of students in each department.

In [5]:
# GroupBy and Aggregate
summary_df = df.groupBy("Department") \
               .agg(
                   avg("GPA").alias("Average_GPA"),
                   count("StudentID").alias("Student_Count")
               )

print("Department Summary:")
summary_df.show()

Department Summary:
+--------------------+-----------+-------------+
|          Department|Average_GPA|Student_Count|
+--------------------+-----------+-------------+
|    Computer Science|        3.7|            2|
|Information Techn...|       3.05|            2|
|Business Administ...|        3.7|            2|
+--------------------+-----------+-------------+



## 5. Running Spark SQL Queries
We can register the DataFrame as a temporary view to run SQL queries directly.

In [6]:
# Register Temp View
df.createOrReplaceTempView("students")

# Run SQL Query
sql_result = spark.sql("""
    SELECT Department, MAX(GPA) as Max_GPA 
    FROM students 
    GROUP BY Department
""")

print("SQL Query Result (Max GPA by Department):")
sql_result.show()

SQL Query Result (Max GPA by Department):
+--------------------+-------+
|          Department|Max_GPA|
+--------------------+-------+
|    Computer Science|    3.8|
|Information Techn...|    3.2|
|Business Administ...|    3.9|
+--------------------+-------+



## 6. Stop Spark Session
Always remember to release resources by stopping the Spark session when finished.

In [7]:
# Stop the Spark session
spark.stop()
print("Spark Session stopped successfully.")

Spark Session stopped successfully.
